# Example 15.4 / 1: Compare Two Answers to One Clarifying Question

Companion notebook for **15.4 AMPLbot in Gemini** in the AMPL lab Documentation, section "Compare Two Answers to One Clarifying Question". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Compare Two Answers to One Clarifying Question

The unresolved choice in the brief changes the decision as well as the cost. To check a model
the Gem returns, run it on a small instance whose answer you can work out by hand. The model below
uses four quarters of demand, a base capacity of 100 tons per quarter, and expansion in whole
units of 10 tons at 500 dollars per unit for the whole horizon. Unmet demand is allowed at 20
dollars per ton through a shortage variable `Short`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set QUARTERS;                       # planning quarters
param demand {QUARTERS} >= 0;       # tons
param base_cap >= 0;                # tons per quarter
param unit_size > 0;                # tons per expansion unit
param expand_cost >= 0;             # dollars per unit
param prod_cost >= 0;               # dollars per ton
param penalty >= 0;                 # dollars per ton short

var Produce {QUARTERS} >= 0;  # tons produced per quarter
var Short {QUARTERS} >= 0;    # unmet demand, tons
var Expand integer >= 0;      # whole expansion units, for the horizon

minimize Total_Cost:  # expansion, production and shortage cost, dollars
    expand_cost * Expand
    + sum {q in QUARTERS} (prod_cost * Produce[q] + penalty * Short[q]);

# base plus expanded capacity, every quarter
subject to Capacity {q in QUARTERS}: Produce[q] <= base_cap + unit_size * Expand;
# shortfall allowed at the penalty
subject to Meet {q in QUARTERS}: Produce[q] + Short[q] >= demand[q];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["QUARTERS"] = ["Q1", "Q2", "Q3", "Q4"]
ampl.param["demand"] = {"Q1": 80, "Q2": 120, "Q3": 100, "Q4": 90}
ampl.param["base_cap"] = 100
ampl.param["unit_size"] = 10
ampl.param["expand_cost"] = 500
ampl.param["prod_cost"] = 2
ampl.param["penalty"] = 20

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.obj["Total_Cost"].value())
print("Expand", ampl.var["Expand"].value())
print(ampl.var["Short"].to_dict())

solved 1140.0
Expand 0.0
{'Q1': 0, 'Q2': 20, 'Q3': 0, 'Q4': 0}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set QUARTERS := Q1 Q2 Q3 Q4;
param demand := Q1 80  Q2 120  Q3 100  Q4 90;
param base_cap := 100;
param unit_size := 10;
param expand_cost := 500;
param prod_cost := 2;
param penalty := 20;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display solve_result, Total_Cost, Expand;
display Short;

solve_result = solved
Total_Cost = 1140
Expand = 0

Short [*] :=
Q1   0
Q2  20
Q3   0
Q4   0
;



### Another way in amplpy

To forbid unmet demand instead, keep the same model.mod and data.py and fix every `Short` at 0
before solving. AMPL's `fix` statement has no amplpy method for an indexed variable, so run.py
uses `ampl.eval`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["QUARTERS"] = ["Q1", "Q2", "Q3", "Q4"]
ampl.param["demand"] = {"Q1": 80, "Q2": 120, "Q3": 100, "Q4": 90}
ampl.param["base_cap"] = 100
ampl.param["unit_size"] = 10
ampl.param["expand_cost"] = 500
ampl.param["prod_cost"] = 2
ampl.param["penalty"] = 20

In [11]:
# run.py
ampl.eval("fix {q in QUARTERS} Short[q] := 0;")
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.obj["Total_Cost"].value())
print("Expand", ampl.var["Expand"].value())

solved 1780.0
Expand 2.0
